# 03b - Heavy metrics (BERTScore, LLM-Score Gemini judge)

**Mục tiêu:** chấm những metrics nặng, tốn thời gian (tốn VRAM/hoặc API), tách riêng để tránh Kaggle kernel die.

**Inputs**
- `PRED_CSV`: file dự đoán từ notebook 02 (detail-only)
- (cho Gemini judge) HF dataset `bbdontcry/vietnamese-image-captioning` để lấy ảnh gốc theo `id`/index
- (tuỳ chọn) `GEMINI_API_KEY` trong Kaggle Secrets + `LLM_SCORE_MODE=on`

**Outputs** *(trong `OUT_DIR`)*  
- `{RUN_ID}_metrics_heavy.json` : `{bertscore_f1, llm_score_mean, n, model_name, ...}`
- `llm_score_cache.json` : cache per-sample để resume (nếu bật Gemini)
- `{RUN_ID}_main_results_row_heavy.csv`

**Notebook kế tiếp:** chạy `04-qwen-vl-merge-results.ipynb` để merge light + heavy.


## 0) Config

In [13]:
import os, torch
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# === USER CONFIG ===
RUN_ID  = "B"  # baseline | A | B
PRED_CSV = f"/kaggle/input/vn-textbook-qwen2vl-02-predictions/{RUN_ID}_predictions_test_detail.csv"
OUT_DIR  = "/kaggle/working"

# name hiển thị trong bảng kết quả
MODEL_NAME = f"Qwen2-VL-2B ({RUN_ID})"

# HuggingFace dataset để lấy ảnh cho LLM-Score
DATASET_ID = "bbdontcry/vietnamese-image-captioning"
DATASET_SPLIT = "test"
DATASET_ID_COL = "id"     # cột id trong dataset HF
DATASET_IMG_COL = "image" # cột ảnh trong dataset HF

# LLM-Score (Gemini)
LLM_SCORE_MODE = os.getenv("LLM_SCORE_MODE", "on")  # "on" to enable Gemini judge, "off" to disable
GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY")

# Gemini model
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "models/gemini-3-flash-preview")

# Parallel Gemini judge
LLM_SCORE_WORKERS = int(os.getenv("LLM_SCORE_WORKERS", "10"))          # số request song song
LLM_SCORE_SAVE_EVERY = int(os.getenv("LLM_SCORE_SAVE_EVERY", "5"))    # ghi cache mỗi N kết quả
LLM_SCORE_RETRY_INVALID = os.getenv("LLM_SCORE_RETRY_INVALID", "0").lower() in ("1","true","yes","on")
LLM_SCORE_MIN_INTERVAL_SEC = float(os.getenv("LLM_SCORE_MIN_INTERVAL_SEC", "0"))  # 0 = không rate-limit toàn cục
LLM_SCORE_BASE_DELAY_SEC = float(os.getenv("LLM_SCORE_BASE_DELAY_SEC", "8"))      # backoff khi 429

LLM_SCORE_MAX_SAMPLES = 0  # 0 = all
GEMINI_MAX_SIDE = 768
LLM_SCORE_MAX_OUTPUT_TOKENS = 8192
LLM_SCORE_SLEEP_SEC = 0.5
LLM_SCORE_CACHE_PATH = os.getenv("LLM_SCORE_CACHE_PATH", f"/kaggle/working/{RUN_ID}_llm_score_cache.json")
LLM_SCORE_MAX_CAND_CHARS = 3500

# BERTScore
BERTSCORE_MODEL_TYPE = "bert-base-multilingual-cased"
BERTSCORE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BERTSCORE_BATCH_SIZE = 2 if BERTSCORE_DEVICE=="cuda" else 8


## 1) Install deps (heavy metrics)

In [15]:
!pip install -q -U bert-score pandas tqdm google-generativeai datasets

## 2) Load predictions

In [16]:
import pandas as pd
import numpy as np

pred_df = pd.read_csv(PRED_CSV)
print("Loaded:", pred_df.shape)
print("Columns:", pred_df.columns.tolist())

def pick_col(cands):
    for c in cands:
        if c in pred_df.columns:
            return c
    return None

COL_PRED = pick_col(["pred_detail", "pred", "prediction", "pred_text"])
COL_REF  = pick_col(["gt_detail", "reference", "ref", "gt_text"])
COL_ID   = pick_col(["id", "sample_id"])

assert COL_PRED is not None, "Missing prediction column"
assert COL_REF  is not None, "Missing reference column"

preds = pred_df[COL_PRED].fillna("").astype(str).tolist()
refs  = pred_df[COL_REF ].fillna("").astype(str).tolist()

n = len(preds)
if LLM_SCORE_MAX_SAMPLES and LLM_SCORE_MAX_SAMPLES > 0:
    n = min(n, LLM_SCORE_MAX_SAMPLES)
    preds = preds[:n]
    refs  = refs[:n]
    pred_df = pred_df.iloc[:n].reset_index(drop=True)

print("Using:", {"pred": COL_PRED, "ref": COL_REF, "id": COL_ID})
print("n =", len(preds))

Loaded: (125, 6)
Columns: ['id', 'gt_detail', 'pred_detail', 'mode', 'model_id', 'adapter_dir']
Using: {'pred': 'pred_detail', 'ref': 'gt_detail', 'id': 'id'}
n = 125


## 3) Prepare images for LLM-Score (Gemini)

In [17]:
from datasets import load_dataset
from PIL import Image

test_ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
print("Loaded HF dataset:", DATASET_ID, DATASET_SPLIT, "len=", len(test_ds))

id_to_idx = None
if COL_ID is not None and DATASET_ID_COL in test_ds.column_names:
    id_to_idx = {str(test_ds[i][DATASET_ID_COL]): i for i in range(len(test_ds))}
    missing = 0
    for i in range(len(pred_df)):
        sid = str(pred_df.loc[i, COL_ID])
        if sid not in id_to_idx:
            missing += 1
    if missing:
        print(f"⚠ Warning: {missing}/{len(pred_df)} ids not found in HF test split. Will fall back to row index.")
else:
    print("⚠ No id column available; will assume prediction rows align with HF test split order.")

def _prepare_image(img: Image.Image, max_side: int = 768) -> Image.Image:
    img = img.convert("RGB")
    w, h = img.size
    m = max(w, h)
    if m <= max_side:
        return img
    scale = max_side / m
    nw, nh = int(w * scale), int(h * scale)
    return img.resize((nw, nh))

def get_image_for_row(i: int) -> Image.Image:
    if id_to_idx is not None:
        try:
            sid = str(pred_df.loc[i, COL_ID])
            j = id_to_idx.get(sid, None)
            if j is None:
                j = i
        except Exception:
            j = i
    else:
        j = i
    img = test_ds[j][DATASET_IMG_COL]
    return _prepare_image(img, GEMINI_MAX_SIDE)

# quick check
img0 = get_image_for_row(0)
print("img0:", type(img0), img0.size)

Loaded HF dataset: bbdontcry/vietnamese-image-captioning test len= 125
img0: <class 'PIL.Image.Image'> (547, 768)


## 4) Metrics: BERTScore (↑) — chunked to avoid kernel die

In [18]:
import os, gc
import torch
from bert_score import score as bert_score_score
from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = BERTSCORE_DEVICE
bs = max(1, int(BERTSCORE_BATCH_SIZE))
model_type = BERTSCORE_MODEL_TYPE

f1_chunks = []
with torch.no_grad():
    for s in tqdm(range(0, len(preds), bs), desc=f"BERTScore ({device}, bs={bs})"):
        p = preds[s:s+bs]
        r = refs[s:s+bs]
        P, R, F1 = bert_score_score(
            p, r,
            lang="vi",
            model_type=model_type,
            device=device,
            verbose=False,
            rescale_with_baseline=False,
        )
        f1_chunks.append(F1.detach().cpu())

bert_f1 = float(torch.cat(f1_chunks).mean().item())
print("BERTScore(F1):", bert_f1)

del f1_chunks
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

BERTScore (cuda, bs=2):   0%|          | 0/63 [00:00<?, ?it/s]

BERTScore(F1): 0.6705487966537476


## 5) Metrics: LLM-Score (Gemini judge, 0..10) — optional (batched + cache)
Bật bằng `LLM_SCORE_MODE=on` + set `GEMINI_API_KEY`.
Notebook sẽ cache vào `LLM_SCORE_CACHE_PATH` để resume.

In [19]:
import os, json, time, re, threading
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

llm_score_mean = None
llm_visual_mean = None
llm_nav_mean = None
llm_valid_n = 0
llm_total_n = 0

if LLM_SCORE_MODE.lower() != "on":
    print("LLM_SCORE_MODE=off -> skip Gemini judge.")
else:
    assert GEMINI_API_KEY, "Missing GEMINI_API_KEY (or GOOGLE_API_KEY) in environment."

    import google.generativeai as genai
    from google.generativeai.types import HarmCategory, HarmBlockThreshold

    genai.configure(api_key=GEMINI_API_KEY)

    SAFETY_SETTINGS = {
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    }

    # ======= PROMPTS =======
    SYSTEM_PROMPT = """Bạn là giám khảo chấm điểm cho dự án "Sách giáo khoa cho người khiếm thị".
Nhiệm vụ: Quan sát HÌNH ẢNH GỐC và chấm điểm độ chính xác của đoạn MÔ TẢ (Candidate).

BẠN PHẢI CHẤM ĐIỂM VÀ GIẢI THÍCH DỰA TRÊN SỰ THẬT TRONG ẢNH (REFERENCE-FREE).

TRỤC 1: VISUAL REALITY CHECK
- 1-2: Ảo giác, bịa vật thể/hành động không có.
- 3-4: Sai lệch lớn chủ thể chính.
- 5-6: Đúng chung nhưng sai màu/số lượng/vị trí.
- 7-8: Khá đúng, có vài thiếu/sai nhỏ.
- 9-10: Rất đúng, sát ảnh, không ảo giác.

TRỤC 2: NAVIGATION & USEFULNESS
- 1-2: Mô tả vô dụng/mơ hồ; không giúp hiểu bố cục.
- 3-4: Thiếu nhiều thông tin quan trọng, khó hình dung.
- 5-6: Đủ cơ bản nhưng thiếu chi tiết/bố cục.
- 7-8: Tốt, mô tả rõ bố cục & điểm chính.
- 9-10: Xuất sắc, dẫn dắt nghe hiểu bố cục/chi tiết quan trọng.

OUTPUT JSON (snake_case) theo schema:
{
  "visual_score": int (1..10),
  "visual_reason": string (<= 180 ký tự),
  "visual_error_type": string (<= 40 ký tự),
  "navigation_score": int (1..10),
  "navigation_reason": string (<= 180 ký tự),
  "navigation_error_type": string (<= 40 ký tự)
}
Chỉ trả JSON, không markdown, không text thừa.
"""

    USER_TEMPLATE = """### DỮ LIỆU
Candidate: "{cand}"

### YÊU CẦU
1) Xem ảnh.
2) Chấm điểm Visual & Navigation.
3) Giải thích ngắn gọn (chỉ ra chỗ sai chính).
4) Trả JSON đúng schema.
"""

    # Fallback prompt: cực ngắn để tránh MAX_TOKENS / output rỗng
    SYSTEM_PROMPT_MIN = "Chỉ chấm điểm 2 trục visual_score và navigation_score (1..10) dựa trên ảnh và candidate. Trả JSON đúng schema, không text thừa."
    USER_TEMPLATE_MIN = "Candidate: {cand}\nTrả JSON: {{\"visual_score\": int, \"navigation_score\": int}}"

    # ======= Generation configs =======
    GEN_CONFIG = {
        "temperature": 0.0,
        "max_output_tokens": int(LLM_SCORE_MAX_OUTPUT_TOKENS),
        "response_mime_type": "application/json",
    }
    GEN_CONFIG_MIN = {
        "temperature": 0.0,
        "max_output_tokens": 256,
        "response_mime_type": "application/json",
    }

    def sanitize_json_string(s: str) -> str:
        if not s:
            return s
        s = re.sub(r"```json\s*", "", s, flags=re.IGNORECASE)
        s = re.sub(r"```\s*$", "", s)
        s = s.strip()
        # extract first {...} block
        m = re.search(r"\{[\s\S]*\}", s)
        if m:
            s = m.group(0)
        # truncate to last brace
        last = s.rfind("}")
        if last != -1:
            s = s[: last + 1]
        return s.strip()

    def safe_text_and_finish(resp):
        # robust extraction (avoid resp.text crash when no Part)
        try:
            return resp.text, None
        except Exception:
            try:
                cand = resp.candidates[0]
                finish_reason = getattr(cand, "finish_reason", None)
                if getattr(cand, "content", None) and getattr(cand.content, "parts", None):
                    txt = "".join([getattr(p, "text", "") for p in cand.content.parts if getattr(p, "text", None)]).strip()
                else:
                    txt = ""
                return txt, finish_reason
            except Exception:
                return "", None

    def clamp_int(x, lo, hi, default=None):
        try:
            v = int(x)
            return max(lo, min(hi, v))
        except Exception:
            return default

    def parse_full_json(txt: str):
        txt = sanitize_json_string(txt or "")
        if not txt:
            raise ValueError("empty_json")
        data = json.loads(txt)
        out = {
            "visual_score": clamp_int(data.get("visual_score"), 1, 10, None),
            "visual_reason": str(data.get("visual_reason", "")).strip()[:180],
            "visual_error_type": str(data.get("visual_error_type", "")).strip()[:40],
            "navigation_score": clamp_int(data.get("navigation_score"), 1, 10, None),
            "navigation_reason": str(data.get("navigation_reason", "")).strip()[:180],
            "navigation_error_type": str(data.get("navigation_error_type", "")).strip()[:40],
        }
        if out["visual_score"] is None or out["navigation_score"] is None:
            raise ValueError("missing_scores")
        out["final_score"] = (out["visual_score"] + out["navigation_score"]) / 2.0
        return out

    def parse_min_json(txt: str):
        txt = sanitize_json_string(txt or "")
        if not txt:
            raise ValueError("empty_json")
        data = json.loads(txt)
        vs = clamp_int(data.get("visual_score"), 1, 10, None)
        ns = clamp_int(data.get("navigation_score"), 1, 10, None)
        if vs is None or ns is None:
            raise ValueError("missing_scores")
        return {
            "visual_score": vs,
            "visual_reason": "",
            "visual_error_type": "",
            "navigation_score": ns,
            "navigation_reason": "",
            "navigation_error_type": "",
            "final_score": (vs + ns) / 2.0,
            "_fallback": "min_schema",
        }

    # ---------- thread-local models + optional global rate limiter ----------
    _thread_local = threading.local()
    _img_lock = threading.Lock()

    class _GlobalRateLimiter:
        def __init__(self, min_interval_sec: float):
            self.min_interval = float(min_interval_sec)
            self._lock = threading.Lock()
            self._next_t = 0.0
        def wait(self):
            if self.min_interval <= 0:
                return
            with self._lock:
                now = time.time()
                sleep = max(0.0, self._next_t - now)
                self._next_t = max(self._next_t, now) + self.min_interval
            if sleep > 0:
                time.sleep(sleep)

    _rate_limiter = _GlobalRateLimiter(LLM_SCORE_MIN_INTERVAL_SEC)

    def _make_model(system_prompt: str, gen_config: dict):
        return genai.GenerativeModel(
            model_name=GEMINI_MODEL,
            system_instruction=system_prompt,
            safety_settings=SAFETY_SETTINGS,
            generation_config=gen_config,
        )

    # sanity check once (fail fast if API/model invalid)
    _ = _make_model(SYSTEM_PROMPT_MIN, GEN_CONFIG_MIN).generate_content("ping")
    print("Using Gemini model:", GEMINI_MODEL)

    def _get_models():
        if not hasattr(_thread_local, "judge_model"):
            _thread_local.judge_model = _make_model(SYSTEM_PROMPT, GEN_CONFIG)
            _thread_local.judge_model_min = _make_model(SYSTEM_PROMPT_MIN, GEN_CONFIG_MIN)
        return _thread_local.judge_model, _thread_local.judge_model_min

    # ---------- cache + todo ----------
    cache_path = Path(LLM_SCORE_CACHE_PATH)
    cache = json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.exists() else {}

    def _key(i: int) -> str:
        if COL_ID is not None:
            return str(pred_df.loc[i, COL_ID])
        return f"idx_{i}"

    MAX_CAND_CHARS = int(LLM_SCORE_MAX_CAND_CHARS)
    def _truncate_cand(s: str) -> str:
        s = (s or "").strip()
        if len(s) <= MAX_CAND_CHARS:
            return s
        return s[:MAX_CAND_CHARS] + " ..."

    def _is_done(i: int) -> bool:
        k = _key(i)
        if k not in cache:
            return False
        if not LLM_SCORE_RETRY_INVALID:
            return True
        return cache.get(k, {}).get("final_score", None) is not None

    all_idxs = list(range(len(pred_df)))
    todo = [i for i in all_idxs if not _is_done(i)]
    llm_total_n = len(all_idxs)
    print(f"LLM-Score: total={len(all_idxs)}, cached={len(all_idxs)-len(todo)}, todo={len(todo)} (workers={LLM_SCORE_WORKERS})")

    def judge_one(i: int):
        cand = _truncate_cand(preds[i])

        # HF dataset access (PIL decode) guarded
        with _img_lock:
            img = get_image_for_row(i)

        # Try full schema first
        prompt = USER_TEMPLATE.format(cand=cand)
        last_err = None
        for attempt in range(1, 4):
            try:
                _rate_limiter.wait()
                jm, _ = _get_models()
                resp = jm.generate_content([img, prompt])
                txt, finish_reason = safe_text_and_finish(resp)
                if not (txt and txt.strip()):
                    raise RuntimeError(f"Empty Gemini output (finish_reason={finish_reason})")
                return parse_full_json(txt)
            except Exception as e:
                last_err = e
                err = str(e)
                if ("429" in err) or ("TooManyRequests" in err) or ("RESOURCE_EXHAUSTED" in err):
                    wait = float(LLM_SCORE_BASE_DELAY_SEC) + attempt * 6.0
                else:
                    wait = float(LLM_SCORE_SLEEP_SEC) + attempt * 1.0
                time.sleep(wait)

        # Fallback: minimal schema (scores only)
        prompt2 = USER_TEMPLATE_MIN.format(cand=cand)
        for attempt in range(1, 3):
            try:
                _rate_limiter.wait()
                _, jm_min = _get_models()
                resp = jm_min.generate_content([img, prompt2])
                txt, finish_reason = safe_text_and_finish(resp)
                if not (txt and txt.strip()):
                    raise RuntimeError(f"Empty Gemini output (finish_reason={finish_reason})")
                return parse_min_json(txt)
            except Exception as e:
                last_err = e
                err = str(e)
                if ("429" in err) or ("TooManyRequests" in err) or ("RESOURCE_EXHAUSTED" in err):
                    wait = float(LLM_SCORE_BASE_DELAY_SEC) + attempt * 6.0
                else:
                    wait = float(LLM_SCORE_SLEEP_SEC) + attempt * 1.0
                time.sleep(wait)

        raise RuntimeError(last_err)

    def _error_result(e: Exception):
        return {
            "visual_score": None,
            "visual_reason": f"SYSTEM ERROR: {str(e)[:200]}",
            "visual_error_type": "System Error",
            "navigation_score": None,
            "navigation_reason": "N/A",
            "navigation_error_type": "N/A",
            "final_score": None,
        }

    def _worker(i: int):
        try:
            res = judge_one(i)
            return _key(i), res
        except Exception as e:
            return _key(i), _error_result(e)

    # ---------- run parallel + periodic save ----------
    if not todo:
        print("LLM-Score: nothing to do (all cached).")
    else:
        updated = 0
        with ThreadPoolExecutor(max_workers=max(1, int(LLM_SCORE_WORKERS))) as ex:
            futures = [ex.submit(_worker, i) for i in todo]
            for fut in tqdm(as_completed(futures), total=len(futures),
                            desc=f"LLM-Score (Gemini, parallel workers={LLM_SCORE_WORKERS})"):
                k, res = fut.result()
                cache[k] = res
                updated += 1
                if updated % int(LLM_SCORE_SAVE_EVERY) == 0:
                    cache_path.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")

        cache_path.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")

    # ---------- aggregate (ignore invalid) ----------
    finals, vs, ns = [], [], []
    invalid = 0
    for i in range(len(pred_df)):
        k = _key(i)
        if k in cache:
            fs = cache[k].get("final_score", None)
            if fs is None:
                invalid += 1
                continue
            finals.append(float(fs))
            vs.append(float(cache[k].get("visual_score", 0.0)))
            ns.append(float(cache[k].get("navigation_score", 0.0)))

    llm_valid_n = len(finals)
    llm_score_mean = float(sum(finals)/len(finals)) if finals else None
    llm_visual_mean = float(sum(vs)/len(vs)) if vs else None
    llm_nav_mean = float(sum(ns)/len(ns)) if ns else None

    print("LLM-Score mean:", llm_score_mean, f"(valid={llm_valid_n}/{llm_total_n}, invalid={invalid})")
    print(" - visual mean:", llm_visual_mean)
    print(" - navigation mean:", llm_nav_mean)


Using Gemini model: models/gemini-3-flash-preview
LLM-Score: total=125, cached=0, todo=125 (workers=5)


LLM-Score (Gemini, parallel workers=5):   0%|          | 0/125 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6) Save outputs (heavy metrics)

In [ ]:
import json
from pathlib import Path
import pandas as pd

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

metrics_heavy = {
    "run_id": RUN_ID,
    "model_name": MODEL_NAME,
    "n": len(preds),
    "bertscore_f1": float(bert_f1),
    "llm_score_mean": (float(llm_score_mean) if llm_score_mean is not None else None),
    "llm_visual_mean": (float(llm_visual_mean) if llm_visual_mean is not None else None),
    "llm_nav_mean": (float(llm_nav_mean) if llm_nav_mean is not None else None),
    "llm_valid_n": int(llm_valid_n),
    "llm_total_n": int(llm_total_n),
}

(out_dir / f"{RUN_ID}_metrics_heavy.json").write_text(
    json.dumps(metrics_heavy, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Wrote:", out_dir / f"{RUN_ID}_metrics_heavy.json")

row = {
    "Model": MODEL_NAME,
    "Quote-CER": "",
    "Concept-Rec": "",
    "LLM-Score": ("" if llm_score_mean is None else float(llm_score_mean)),
    "BERTScore": float(bert_f1),
    "BLEU-4": "",
    "METEOR": "",
}
pd.DataFrame([row]).to_csv(out_dir / f"{RUN_ID}_main_results_row_heavy.csv", index=False)
print("Wrote:", out_dir / f"{RUN_ID}_main_results_row_heavy.csv")


## 7) (Optional) Merge light+heavy into one row
Nếu bạn đã chạy Notebook A và có `metrics_light.json` trong cùng `OUT_DIR`.

In [ ]:
import json
from pathlib import Path
import pandas as pd

out_dir = Path(OUT_DIR)
light_p = Path("/kaggle/input/qwen-vl-metrics/metrics_light_baseline.json")
heavy_p = out_dir / "metrics_heavy.json"

csv_path  = out_dir / 'main_results_row_merged.csv'
json_path = out_dir / 'main_results_row_merged.json'

if light_p.exists() and heavy_p.exists():
    light = json.loads(light_p.read_text(encoding="utf-8"))
    heavy = json.loads(heavy_p.read_text(encoding="utf-8"))

    row = {
        "Model": MODEL_NAME,
        "Quote-CER": light.get("quote_cer", ""),
        "Concept-Rec": light.get("concept_rec", ""),
        "LLM-Score": heavy.get("llm_score_mean", ""),
        "BERTScore": heavy.get("bertscore_f1", ""),
        "BLEU-4": light.get("bleu4", ""),
        "METEOR": light.get("meteor", ""),
    }

    # CSV
    pd.DataFrame([row]).to_csv(csv_path, index=False)

    # JSON (khuyên dùng: pretty + utf-8, không escape tiếng Việt)
    json_path.write_text(
        json.dumps(row, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print("Wrote:", csv_path)
    print("Wrote:", json_path)
    print(row)

else:
    print("Missing metrics_light.json or metrics_heavy.json in OUT_DIR. Run Notebook A/B first.")

